# Gold - KPI de demanda horaria

**Entrada:** `case.gold.ft_chamados`, produzida por `Gold_ft_chamados_v2.ipynb`.
**Saida:** `case.gold.kpi_demanda_horaria`.
**Grao:** mes de abertura x dia da semana x hora de abertura x canal.

Nao executa joins, nao corrige datas e nao remove chamados. Abertura ausente gera chaves temporais nulas; datas fora do periodo permanecem visiveis e contabilizadas. Grupos sem chamados nao sao criados artificialmente.

As datas ja corrigidas pela Silver sao consumidas como publicadas na fato. As colunas originais de tempo sao da Silver tratada, nao do CSV bruto. Consultar `PENDENCIAS_GOLD.md`.


## 1. Configuracao

Widgets seguem o padrao da Gold da equipe. O limite padrao e 300 segundos (cinco minutos), configuravel. O periodo e inclusivo e serve para sinalizacao, sem filtro. O fuso da sessao sera informado no resumo; nao sera alterado automaticamente.


In [ ]:
import json
import math
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, StringType, TimestampType, TimestampNTZType


def parametro(nome, padrao):
    try:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)
    except NameError:
        return padrao


CATALOG = parametro("catalog", "case")
GOLD_SCHEMA = parametro("gold_schema", "gold")
LIMITE_ESPERA_SEGUNDOS = int(parametro("limite_espera_segundos", "300"))
PRECISAO_MEDIANA = int(parametro("precisao_mediana", "10000"))
PERIODO_INICIO = parametro("periodo_inicio", "2025-01-01")
PERIODO_FIM = parametro("periodo_fim", "2026-07-31")
TABELA_FATO = "ft_chamados"
TABELA_DESTINO = "kpi_demanda_horaria"

if LIMITE_ESPERA_SEGUNDOS < 0 or PRECISAO_MEDIANA <= 0:
    raise ValueError("Limite deve ser >= 0 e precisao da mediana deve ser > 0")
if date.fromisoformat(PERIODO_INICIO) > date.fromisoformat(PERIODO_FIM):
    raise ValueError("Periodo inicial posterior ao final")


def fq(tabela):
    return ".".join("`" + parte.replace("`", "``") + "`"
                    for parte in [CATALOG, GOLD_SCHEMA, tabela])


CHAVES = ["mes_referencia", "dia_semana_num", "hora_abertura", "canal"]


## 2. Contrato e qualidade da entrada

A tabela precisa existir no Databricks; copiar o notebook da fato nao a cria. Exigimos IDs nao nulos e unicos. Nao usamos deduplicacao automatica para mascarar problemas nos joins da fato.

As verificacoes de esquema ocorrem antes dos calculos. A resolucao admite somente `sim`, `nao` ou nulo. Os campos numericos originais sao obrigatorios para manter a possibilidade de comparacao.


In [ ]:
if not spark.catalog.tableExists(fq(TABELA_FATO)):
    raise ValueError("Execute e valide Gold_ft_chamados_v2 antes desta KPI")

ft = spark.table(fq(TABELA_FATO))
COLUNAS = [
    "id_chamado", "canal", "resolvido", "hora_abertura_chamado",
    "hora_inicio_atendimento", "hora_finalizacao_atendimento", "status_consistencia",
    "tempo_espera", "tempo_atendimento", "tempo_espera_original", "tempo_atendimento_original",
]
faltando = set(COLUNAS) - set(ft.columns)
if faltando:
    raise ValueError(f"Contrato da fato incompleto: {sorted(faltando)}")
ft = ft.select(*COLUNAS)
tipos = {campo.name: campo.dataType for campo in ft.schema.fields}
for coluna in ["hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento"]:
    if not isinstance(tipos[coluna], (TimestampType, TimestampNTZType)):
        raise ValueError(f"{coluna} deve ser timestamp, recebido: {tipos[coluna]}")
for coluna in ["tempo_espera", "tempo_atendimento", "tempo_espera_original", "tempo_atendimento_original"]:
    if not isinstance(tipos[coluna], NumericType):
        raise ValueError(f"{coluna} deve ser numerica")
for coluna in ["canal", "resolvido", "status_consistencia"]:
    if not isinstance(tipos[coluna], StringType):
        raise ValueError(f"{coluna} deve ser string")

entrada = ft.agg(
    F.count("*").alias("total"),
    F.countDistinct("id_chamado").alias("ids_unicos"),
    F.count("id_chamado").alias("ids_preenchidos"),
    F.count(F.when(~F.col("resolvido").isin("sim", "nao"), 1)).alias("resolucao_invalida"),
).first().asDict()
if entrada["total"] != entrada["ids_unicos"] or entrada["total"] != entrada["ids_preenchidos"]:
    raise ValueError(f"Fato com chave nula ou duplicada: {entrada}")
if entrada["resolucao_invalida"]:
    raise ValueError("Resolucao fora de sim/nao/nulo na fato")


## 3. Preparacao das duracoes e grupos

- Espera: inicio menos abertura, em segundos.
- Atendimento: finalizacao menos inicio, em segundos.
- Exigimos `status_consistencia = CONSISTENTE`, timestamps necessarios presentes e ausencia de inversoes cronologicas entre os horarios disponiveis. Zero e valido.
- Tempos invalidos ficam nulos nas medidas; o chamado continua no volume.
- Nulo em resolucao nao e convertido em `nao`; e contado separadamente.
- Domingo=1 ate sabado=7, seguindo a fato e dim_tempo existentes.
- Canal nulo/vazio recebe `sem canal`; demais valores sao preservados.

As medidas `_original` permitem comparar com os campos tratados da tabela principal. As contagens de comparacao usam somente pares preenchidos e finitos. Tambem verificamos se os tempos publicados na fato correspondem aos recalculos.


In [ ]:
abertura = F.col("hora_abertura_chamado")
inicio = F.col("hora_inicio_atendimento")
fim = F.col("hora_finalizacao_atendimento")

def verdadeiro(cond):
    return F.coalesce(cond, F.lit(False))


def finito(coluna):
    valor = F.col(coluna).cast("double")
    return valor.isNotNull() & ~F.isnan(valor) & (F.abs(valor) < float("inf"))


inversao = verdadeiro(inicio < abertura) | verdadeiro(fim < inicio) | verdadeiro(fim < abertura)
consistente = verdadeiro(F.col("status_consistencia") == "CONSISTENTE") & ~inversao
espera_valida = consistente & abertura.isNotNull() & inicio.isNotNull()
atendimento_valido = consistente & inicio.isNotNull() & fim.isNotNull()

base = (
    ft.withColumn("mes_referencia", F.trunc(F.to_date(abertura), "month"))
    .withColumn("dia_semana_num", F.dayofweek(abertura))
    .withColumn("hora_abertura", F.hour(abertura))
    .withColumn("canal", F.when(F.col("canal").isNull() | (F.trim("canal") == ""),
                                F.lit("sem canal")).otherwise(F.col("canal")))
    .withColumn("espera_seg", F.when(espera_valida, F.unix_timestamp(inicio) - F.unix_timestamp(abertura)))
    .withColumn("atendimento_seg", F.when(atendimento_valido, F.unix_timestamp(fim) - F.unix_timestamp(inicio)))
    .withColumn("horarios_validos", espera_valida & atendimento_valido)
    .withColumn("abertura_fora_periodo", verdadeiro(
        ~F.to_date(abertura).between(PERIODO_INICIO, PERIODO_FIM)))
)

# null-safe: inclusive a diferenca entre um tempo nulo e um tempo preenchido.
erros_tempos = base.filter(
    ~F.col("tempo_espera").eqNullSafe(F.col("espera_seg"))
    | ~F.col("tempo_atendimento").eqNullSafe(F.col("atendimento_seg"))
).limit(1).count()
if erros_tempos:
    raise ValueError("Tempos da fato divergem das regras/timestamps. Revisar a fato antes de publicar a KPI")


## 4. Agregacao e formulas

`qtd_chamados` conta cada chamado uma vez. Medias = soma de duracoes validas / quantidade valida / 60.

`pct_espera_acima_limite` = 100 x qtd_espera_acima_limite / qtd_espera_valida; a comparacao e **estritamente maior** que o limite. `pct_resolvidos` = 100 x qtd_resolvidos / qtd_chamados. Resolucao ausente continua explicita.

Mediana usa `percentile_approx(..., 0.5, precisao_mediana)` somente sobre esperas validas. E aproximada e nao pode ser somada nem recalculada a partir das medianas dos grupos.

Somas e contagens permitem recalcular medias ponderadas e percentuais no Power BI. Nao tirar media das medias nem somar percentuais. Grupos sem duracoes validas tem medias e mediana nulas. Fonte vazia gera tabela vazia com esquema definido.


In [ ]:
def contar(cond):
    return F.count(F.when(cond, F.lit(1)))


def somar(coluna):
    return F.coalesce(F.sum(coluna), F.lit(0))


AGREGACOES = [
    F.count("*").alias("qtd_chamados"),
    contar(F.col("resolvido") == "sim").alias("qtd_resolvidos"),
    contar(F.col("resolvido").isin("sim", "nao")).alias("qtd_resolucao_informada"),
    contar(F.col("resolvido").isNull()).alias("qtd_resolucao_ausente"),
    F.count("espera_seg").alias("qtd_espera_valida"),
    F.count("atendimento_seg").alias("qtd_atendimento_valido"),
    contar(F.col("horarios_validos")).alias("qtd_horarios_validos"),
    contar(F.col("hora_abertura_chamado").isNull()).alias("qtd_sem_abertura"),
    contar(F.col("hora_inicio_atendimento").isNull()).alias("qtd_sem_inicio"),
    contar(F.col("abertura_fora_periodo")).alias("qtd_abertura_fora_periodo"),
    contar(F.col("status_consistencia").isNull() | (F.col("status_consistencia") != "CONSISTENTE"))
        .alias("qtd_cronologia_nao_consistente"),
    somar("espera_seg").alias("soma_espera_segundos"),
    somar("atendimento_seg").alias("soma_atendimento_segundos"),
    F.percentile_approx("espera_seg", 0.5, PRECISAO_MEDIANA).alias("mediana_espera_segundos"),
    contar(F.col("espera_seg") > LIMITE_ESPERA_SEGUNDOS).alias("qtd_espera_acima_limite"),
]
for medida in ["espera", "atendimento"]:
    original = f"tempo_{medida}_original"
    par = F.col(f"{medida}_seg").isNotNull() & finito(original)
    AGREGACOES.extend([
        contar(par).alias(f"qtd_comparavel_{medida}_original"),
        contar(par & (F.col(f"{medida}_seg") != F.col(original))).alias(f"qtd_divergente_{medida}_original"),
        somar(F.when(par, F.col(original))).alias(f"soma_{medida}_original_comparavel_segundos"),
        somar(F.when(par, F.col(f"{medida}_seg"))).alias(f"soma_{medida}_calculada_comparavel_segundos"),
    ])

kpi = base.groupBy(*CHAVES).agg(*AGREGACOES)
for medida, denominador in [("espera", "qtd_espera_valida"),
                            ("atendimento", "qtd_atendimento_valido")]:
    kpi = kpi.withColumn(f"tempo_medio_{medida}_min", F.when(
        F.col(denominador) > 0,
        F.col(f"soma_{medida}_segundos") / F.col(denominador) / 60.0))
kpi = (
    kpi.withColumn("tempo_mediano_espera_min", F.col("mediana_espera_segundos") / 60.0)
    .withColumn("pct_espera_acima_limite", F.when(F.col("qtd_espera_valida") > 0,
        100.0 * F.col("qtd_espera_acima_limite") / F.col("qtd_espera_valida")))
    .withColumn("pct_resolvidos", F.when(F.col("qtd_chamados") > 0,
        100.0 * F.col("qtd_resolvidos") / F.col("qtd_chamados")))
    .withColumn("limite_espera_segundos", F.lit(LIMITE_ESPERA_SEGUNDOS))
    .withColumn("dia_semana_nome", F.element_at(F.array(*[F.lit(d) for d in
        ["domingo", "segunda-feira", "terca-feira", "quarta-feira", "quinta-feira", "sexta-feira", "sabado"]]),
        F.col("dia_semana_num")))
)


## 5. Portao de qualidade antes da publicacao

Chave agregada unica, contagens nao negativas, denominadores coerentes e percentuais entre 0 e 100. Totais aditivos (contagens e somas) sao reconciliados com a base, sem usar medianas nessa reconciliacao. A conservacao dos chamados tambem e comparada com a entrada.

Essas consultas sao verificacoes obrigatorias, nao debugs opcionais. Nao ha `display()` nem logs por grupo.


In [ ]:
if kpi.groupBy(*CHAVES).count().filter(F.col("count") != 1).limit(1).count():
    raise ValueError("Chave agregada duplicada")

contagens = [c for c in kpi.columns if c.startswith("qtd_")]
invalidos = F.lit(False)
for c in contagens:
    invalidos = invalidos | F.col(c).isNull() | (F.col(c) < 0) | (F.col(c) > F.col("qtd_chamados"))
invalidos = invalidos | (F.col("qtd_chamados") <= 0)
invalidos = invalidos | (F.col("qtd_espera_acima_limite") > F.col("qtd_espera_valida"))
invalidos = invalidos | (F.col("qtd_resolvidos") > F.col("qtd_resolucao_informada"))
invalidos = invalidos | (F.col("qtd_resolucao_informada") + F.col("qtd_resolucao_ausente") != F.col("qtd_chamados"))
invalidos = invalidos | (F.col("qtd_horarios_validos") > F.least("qtd_espera_valida", "qtd_atendimento_valido"))
for medida in ["espera", "atendimento"]:
    invalidos = invalidos | (F.col(f"qtd_divergente_{medida}_original") > F.col(f"qtd_comparavel_{medida}_original"))
for c in ["pct_espera_acima_limite", "pct_resolvidos"]:
    invalidos = invalidos | (F.col(c).isNotNull() & ~F.col(c).between(0, 100))
if kpi.filter(invalidos).limit(1).count():
    raise ValueError("Contagem, denominador ou percentual invalido")

esperado = base.agg(*AGREGACOES).first().asDict()
aditivas = [c for c in esperado if c != "mediana_espera_segundos"]
obtido = kpi.agg(*[somar(c).alias(c) for c in aditivas]).first().asDict()
for c in aditivas:
    if not math.isclose(float(esperado[c] or 0), float(obtido[c] or 0), rel_tol=1e-12, abs_tol=1e-6):
        raise ValueError(f"Reconciliacao falhou em {c}: esperado={esperado[c]}, obtido={obtido[c]}")
if obtido["qtd_chamados"] != entrada["total"]:
    raise ValueError("Total agregado difere da fato")


## 6. Gravacao e ordem de execucao

Executar primeiro a Silver e a fato v2 validada. Depois executar este notebook. A KPI mensal sera independente desta agregacao.

Reconstrucao completa em Delta com `overwrite` e `overwriteSchema`: mesma fonte e mesmos parametros geram os mesmos indicadores, sem duplicar linhas. Nao adicionamos timestamp variavel aos dados agregados. Validar o resultado real no Databricks antes de integrar a branch.


In [ ]:
(
    kpi.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(fq(TABELA_DESTINO))
)
resumo = {
    "tabela": fq(TABELA_DESTINO),
    "qtd_chamados": entrada["total"],
    "qtd_espera_valida": obtido["qtd_espera_valida"],
    "qtd_atendimento_valido": obtido["qtd_atendimento_valido"],
    "qtd_sem_abertura": obtido["qtd_sem_abertura"],
    "qtd_abertura_fora_periodo": obtido["qtd_abertura_fora_periodo"],
    "limite_espera_segundos": LIMITE_ESPERA_SEGUNDOS,
    "fuso_sessao": spark.conf.get("spark.sql.session.timeZone"),
}
print(json.dumps(resumo, ensure_ascii=False))
try:
    dbutils.notebook.exit(json.dumps(resumo, ensure_ascii=False))
except NameError:
    pass
